# **UCI Heart Disease Cleveland Dataset Project**

**By Group A:**

- Anas Ben Zitun
- Marwan Elamami
- Sanad Ben Latif
- Abdul Majed Mraied

> **Research limitations:** This is a historical educational analysis, not a clinically validated tool. Feature selection and imputation use the full dataset before splitting, and models are compared on the test set. These choices can bias evaluation. See the README before interpreting results.

### **Project Overview:**

Cardiovascular diseases are the leading cause of death worldwide, claiming millions of lives annually. Early diagnosis of these diseases presents a significant challenge in the medical field, often relying on complex clinical examinations and tests that can be costly or unavailable in some areas. This is where data science plays a revolutionary role in supporting medical decisions by analyzing clinical data and extracting patterns that help predict risk.

In this project, machine learning techniques were used on heart disease data from the renowned Cleveland Clinic to build a model capable of predicting the presence of the disease based on a range of clinical indicators, such as the type of chest pain, heart rate, thallium test results, and others. This approach is not intended to replace physicians, but rather to provide an aid that improves diagnostic efficiency and reduces instances of medical negligence.



---

---

# **1. Introduction & Data Overview:**
* The project leverages the Cleveland Heart Disease dataset, the most widely used subset of the UCI database.

* Out of 76 attributes, research focuses on 14 key clinical features for prediction.

* The task is framed as binary classification: Target = 0 (no disease) vs. Target > 0 (disease present).

## **1.1. Feature Definitions**

### **1.1.1. Top Predictive Features**

These variables were identified as having the strongest correlation and visual relationship with heart disease severity (`num`)


| Old Name | New Name for EDA | Description | Importance | Preprocessing Note |
| :--- | :--- | :--- | :--- | :--- |
| **`num`** | **`Target`** | **Target Variable: Disease Severity** | **Target** | Multi-class (0-4) classification outcome. |
| **`ca`** | **`BlockedVessels`** | **Number of Major Vessels Blocked** | **Very High** | Strongest structural predictor ($0$ to $3$ major vessels colored by fluoroscopy), contains missing values that require imputation |
| **`thalach`** | **`MaxHR`** | **Max Heart Rate Achieved** | **High** | Strong physiological predictor, Inversely correlated with severity. Needs scaling due to distribution/outliers |
| **`oldpeak`** | **`STDepression`** | **ST Depression** | **High** | Exercise-induced ST depression relative to rest, needs scaling due to positive skew and outliers |
| **`thal`** | **`ThalResult`** | **Thallium Stress Test Result** | **High** | Indicator of blood flow to the heart (3=Normal, 6: Fixed Defect, 7: Reversible Defect), contains missing values that require imputation |
| **`cp`** | **`ChestPain`** | **Chest Pain Type** | **Medium** | Crucial categorical feature distinguishing symptomatic pain types (1-3) from asymptomatic risk (4) |

**NOTE:** this section was added after the EDA was done

### **1.1.2. Holistic Feature Dictionary**
This table provides definitions for all 14 features in the dataset

| Old Name | New Name for EDA | Description | Type | Value Range & Meaning |
| :--- | :--- | :--- | :--- | :--- |
| `age` | `Age` | Age of patient | Continuous | Years |
| `sex` | `Sex` | Gender | Categorical | 1 = Male, 0 = Female |
| `cp` | `ChestPain` | Chest Pain Type | Categorical | 1: Typical Angina, 2: Atypical Angina, 3: Non-Anginal Pain, 4: Asymptomatic |
| `trestbps` | `RestBP` | Resting Blood Pressure | Continuous | mm Hg on admission to the hospital |
| `chol` | `Cholesterol` | Serum Cholesterol | Continuous | mg/dl |
| `fbs` | `FastingBS` | Fasting Blood Sugar | Categorical | 1: > 120 mg/dl (True), 0: $\le$ 120 mg/dl (False) |
| `restecg` | `RestECG` | Resting Electrocardiographic Results | Categorical | 0: Normal, 1: ST-T wave abnormality, 2: Left ventricular hypertrophy |
| `thalach` | `MaxHR` | Maximum Heart Rate Achieved | Continuous | beats/min |
| `exang` | `ExAngina` | Exercise Induced Angina | Categorical | 1 = Yes, 0 = No |
| `oldpeak` | `STDepression` | ST Depression induced by exercise relative to rest | Continuous | Numeric value (mm) |
| `slope` | `STSlope` | The slope of the peak exercise ST segment | Categorical | 1: Upsloping, 2: Flat, 3: Downsloping |
| `ca` | `BlockedVessels` | Number of major vessels (0-3) colored by flourosopy | Categorical/Ordinal | 0, 1, 2, 3 (Structural Damage Measure) |
| `thal` | `ThalResult` | Thallium Stress Test Result | Categorical | 3: Normal, 6: Fixed Defect, 7: Reversible Defect |
| **`num`** | **`Target`** | **Diagnosis of heart disease (Angiographic Disease Status)** | **Target** | 0: No disease, 1-4: Disease progression |

## **1.2. Data Loading**
&nbsp;&nbsp;&nbsp;&nbsp; first we need to import tools and load the dataset to understand its dimensions

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from testcases import (
    test_pipeline_used,
    test_column_transformer_used,
    test_categorical_encoding,
    test_feature_scaling,
    test_no_data_leakage,
    test_train_test_split,
    test_classification_metrics,
    test_regression_metrics
)

# Suppress all warnings to clean up output
warnings.filterwarnings('ignore')

# Set professional visualization styles
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

# Load the dataset
df = pd.read_csv('heart_disease.csv')

## **1.3. Initial Inspection**

In [ ]:
display(df.head())

print("\n","="*100)
print()

df.info()

print("\n","="*100)
print()

display(df.describe().T)

**Insight:**

- The dataset contains 303 entries, with `BlockedVessels` (`ca`) having 4 missing entries ($1.3\%$ of the dataset) and `ThalResult` (`thal`) having 2 missing entries ($0.66\%$ of the dataset)

- The mean age is approximately $54.4$ years, ranging from $29$ to $77$, this tells us that the dataset focuses on middle-aged to elderly population (high risk people for cardiovascular issues)

- The `Sex` mean value (~ $0.68$) suggests that $68\%$ of our dataset are males, we should mindful of this gender imbalance

- The average of `RestBP` is $131 \text{ mmHg}$ which is slightly above the healthy threshold ($120 \text{ mmHg}$), the max value is $200 \text{ mmHg}$ which indicates that we might have hypertensive patients in this dataset

- The mean in `Cholesterol` is $246 \text{ mg/dl}$, which is **High**, this confirms that our dataset consists of high-risk population, which makes sense but also means that `Cholesterol` might be a noisy predictor, because even many healthy people in this dataset have  high `Cholesterol`, with maximum value of $564 \text{ mg/dl}$, we clearly have outliers that may require scaling or capping before modeling

- The average in `MaxHR` is $149 \text{bpm}$, but notice the range $(\text{71 to 202})$, low maximum heart rate is often a red flag for heart disease

- The median (50% column in the table) in `STDepression` is only $0.8$, but the maximum value is $6.2$, which means that the heart is struggling to get oxygen during exercise, but since most healthy people have ST depression that is near zero, those values (outliers) are likely our most strongest indicators of disease presence

- The majority of data in `BlockedVessels` (up to the 75th percentile) shows 0, 1, or 2 vessel blocked, the jump to 3 vessels as maximum value represents that severe cases are in this dataset


**Highly Important Notes:**
- Imputation stratgies for missing values is highly recommended

- Features like `STDepression` and `Cholesterol` have means significantly higher than their medians (50%), confirming a positive skew, we will likely need to use `StandardScaler` to normalize these so the outliers don't "confuse" our model

- Outliers are seen as errors to be deleted, **but in medical data they are often the most important data point that indicates a "screaming signal" that this patient has heart disease**

## **1.4. Data Cleaning and Structuring for EDA**

**Note: This isn't the full data cleaning process, just what's needed to explore the data and make conclusions.**

### **1.4.1. Column Renaming**
Rename confusing columns (e.g., cp $\rightarrow$ ChestPain) for better plot labels.

In [ ]:
df.columns = ['Age', 'Sex', 'ChestPain', 'RestBP', 'Cholesterol', 'FastingBS',
              'RestECG', 'MaxHR', 'ExAng', 'STDepression', 'STSlope', 'BlockedVessels', 'ThalResult', 'Target']

### **1.4.2. Handling Missing Values and Checking for Duplicates for EDA**

In [ ]:
df_clean = df.dropna().copy()
df_plot = df_clean.copy()
print(f"Original df rows: {len(df)}, df_clean rows after dropping NaNs: {len(df_clean)}")

# Checking for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicates}")


### **1.4.3. Target Correction**

In [ ]:
df_plot['Target_Label'] = df_plot['Target'].apply(lambda x: 'Disease' if x > 0 else 'No Disease')

### **1.4.4. Categorical Feature Mapping**

In [ ]:
df_plot['Sex'] = df_plot['Sex'].map({0.0: 'Female', 1.0: 'Male'})
df_plot['ChestPain'] = df_plot['ChestPain'].map({1.0: 'Typical Angina', 2.0: 'Atypical Angina', 3.0: 'Non-anginal Pain', 4.0: 'Asymptomatic'})
df_plot['ExAng'] = df_plot['ExAng'].map({0.0: 'No', 1.0: 'Yes'})
df_plot['Thal_Label'] = df_plot['ThalResult'].map({3.0: 'Normal', 6.0: 'Fixed Defect', 7.0: 'Reversible Defect'})
print("Categorical features mapped to human-readable labels.")

### **1.4.5. Defining Feature Lists for Analysis**

In [ ]:
numerical_features = ['Age', 'RestBP', 'Cholesterol', 'MaxHR', 'STDepression']

categorical_features = ['Sex', 'ChestPain', 'FastingBS', 'RestECG', 'ExAng', 'STSlope', 'BlockedVessels', 'Thal_Label', 'Target_Label']

## **2. Univariate Analysis (Analyzing 1 variable at a time)**

We must understand the distribution of your data

### **2.1. Target Variable Analysis**

#### **2.1.1. Heart Disease Severity Distribution**

In [ ]:
df_plot['Disease_Present'] = df_plot['Target'].apply(lambda x: 1 if x > 0 else 0)

plt.figure(figsize=(7, 5))
ax = sns.countplot(x='Target', hue='Target', data=df_plot, palette='viridis', legend=False)


plt.title('Distribution of Heart Disease Severity (0=Healthy, 4=Critical)', fontsize=14)
plt.xlabel('Disease Severity Class (num)', fontsize=12)
plt.ylabel('Patient Count', fontsize=12)
plt.xticks([0, 1, 2, 3, 4], ['0: None', '1: Mild', '2: Moderate', '3: Severe', '4: Critical'])
for container in ax.containers:
    ax.bar_label(container)
plt.show()

#### **2.1.2. Disease Presence Distribution**


In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Disease_Present', hue='Disease_Present', data=df_plot, palette='viridis', legend=False)

plt.title('Distribution of Heart Disease Presence', fontsize=14)
plt.xlabel('Diseased (1) No Disease (0)', fontsize=12)
plt.ylabel('Patient Count', fontsize=12)
plt.xticks([0, 1], ['0: No Disease', '1: Diseased'])

for container in ax.containers:
    ax.bar_label(container)

plt.show()

**Finding:** The dataset is well-balanced for the binary classification task:
 - **No Disease (Healthy):** 160 patients (53.9%)
 - **Disease:** 137 patients (46.1%)

 The target variable distribution shows that the dataset is relatively balanced for binary classification (~54% No Disease, ~46% Disease). However, for multi-class analysis, 'Class 0' (Healthy) is the majority, while higher severity levels (Classes 2, 3, and 4) have fewer samples. This suggests that while predicting presence of disease is straightforward, predicting severity might require more data or specialized techniques

#### **2.2. Numerical Features**


In [ ]:
sns.set_theme(style="whitegrid")

df_plot["Disease_Present"] = df_plot["Disease_Present"].astype(str)

numerical_features_to_plot = ['Age', 'Cholesterol', 'RestBP', 'MaxHR', "STDepression"]

n_rows = len(numerical_features_to_plot)
n_cols = 2

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows), constrained_layout=True)

for i, col in enumerate(numerical_features_to_plot):
    sns.histplot(
        data=df_plot,
        x=col,
        hue="Disease_Present",
        kde=True,
        element="step",
        palette={"0": "steelblue", "1": "crimson"},
        ax=axes[i, 0],
        alpha=0.6
    )
    axes[i, 0].set_title(f'{col} Distribution', fontsize=14, weight="bold")
    axes[i, 0].set_xlabel(col, fontsize=12)
    axes[i, 0].set_ylabel('Patient Count', fontsize=12)
    axes[i, 0].grid(axis='y', linestyle='--', alpha=0.7)

    # Boxplot
    sns.boxplot(
        data=df_plot,
        x="Disease_Present",
        y=col,
        palette={"0": "steelblue", "1": "crimson"},
        ax=axes[i, 1]
    )
    axes[i, 1].set_title(f'{col} Outliers', fontsize=14, weight="bold")
    axes[i, 1].set_xlabel("Disease Presence (0 = No, 1 = Yes)", fontsize=12)
    axes[i, 1].set_ylabel('')
    axes[i, 1].grid(axis='y', linestyle='--', alpha=0.7)

# Overall title
fig.suptitle('Distributions and Outliers of Key Numerical Features', fontsize=16, weight="bold", y=1.02)
plt.show()




**Insight:**



| Feature | Distribution | Clinical Insight | Modeling Note |
| :--- | :--- | :--- | :--- |
| **Age** | Nearly normal, centered around 54–56 years. | Focuses on middle-aged to elderly patients, aligning with the high-risk group for heart disease. | No significant outliers; acts as a risk factor that may interact with other features. |
| **Cholesterol** | Right-skewed, with most values between 200–300 mg/dl. | High levels are common even among healthy individuals, suggesting it might be a noisy predictor. | StandardScaler or log transformation recommended; retain outliers as they may signal severe disease. |
| **RestBP** | Slightly skewed, peaking around 130 mmHg. | Average is slightly above the healthy 120 mmHg threshold, indicating possible hypertension. | May benefit from scaling; a risk factor rather than a definitive predictor. |
| **MaxHR** | Normal, centered around 150 bpm. | Lower MaxHR during exercise is a strong indicator of heart disease; healthy individuals reach higher rates. | Strong negative correlation with disease; should be treated as a primary physiological signal. |
| **STDepression** | Heavily right-skewed, most values below 1 mm. | High ST depression during exercise is a critical marker of myocardial ischemia. | One of the most powerful predictors; requires scaling, and outliers are clinically meaningful. |


**Important Notes:**
* **Scale before modeling:** Age, Cholesterol, RestBP, MaxHR, and STDepression.
* **Keep outliers:** Especially in **STDepression** and **Cholesterol**—they may represent high-risk patients.
* **Focus features:** **MaxHR** and **STDepression** are key drivers for classification.

### **2.3. Categorical Features**


In [ ]:
categorical_cols = ['Sex', 'ChestPain', 'FastingBS', 'RestECG', 'ExAng', 'STSlope', 'ThalResult', 'BlockedVessels']

# Create subplot grid: 3 rows × 3 columns
fig = make_subplots(rows=3, cols=3, subplot_titles=[f"{col}" for col in categorical_cols])

# Loop through each categorical column
for i, col in enumerate(categorical_cols):
    row = (i // 3) + 1
    col_pos = (i % 3) + 1
    counts = df_plot[col].value_counts().sort_index()

    fig.add_trace(
        go.Bar(
            x=counts.index.astype(str),
            y=counts.values,
            marker_color='rgba(100, 150, 255, 0.7)',
            text=counts.values,
            textposition='auto',
            name=col
        ),
        row=row,
        col=col_pos
    )

# Update layout
fig.update_layout(
    height=1300,
    width=1200,
    title_text="Distribution of All Categorical Features",
    title_font_size=22,
    showlegend=False,
    template='plotly_white'
)

fig.update_xaxes(title_font=dict(size=10), tickfont=dict(size=10))
fig.update_yaxes(title_text='Count', title_font=dict(size=10), tickfont=dict(size=10))

fig.show()

**Insight:**

| Feature | Distribution | Clinical Insight | Modeling Note |
| :--- | :--- | :--- | :--- |
| **Sex** | ~68% Male, ~32% Female. | Heart disease is more prevalent in males here, but gender alone isn't a dominant predictor. | Watch for bias; models may overfit to male patterns due to imbalance. |
| **ChestPain (cp)** | Asymptomatic (Type 4) is the most frequent. | Paradoxically, asymptomatic patients are more likely to have disease (silent heart issues). | Strong predictor; requires One-Hot Encoding. |
| **FastingBS (fbs)** | Majority are ≤120 mg/dl (Normal). | High blood sugar (diabetes link) increases risk but is a weak standalone predictor. | Useful when combined with cholesterol and blood pressure features. |
| **RestECG** | Most patients are Normal (Type 0). | Abnormal patterns (ST-T wave or hypertrophy) correlate with disease presence. | Moderate predictive power; requires categorical encoding. |
| **ExAng** | Clear split between Yes/No. | Exercise-induced angina is a major red flag and clinical marker. | Binary feature; highly predictive of disease presence. |
| **STSlope (slope)** | Majority are Flat (Type 2). | Flat or downsloping ST segments are strongly associated with ischemia. | Important categorical predictor for model accuracy. |
| **BlockedVessels (ca)**| Most have 0–2; very few have 3. | One of the strongest predictors; 1–3 blocked vessels almost always indicates disease. | Ordinal feature; can be treated as numeric or categorical. |
| **ThalResult (thal)** | Most are Normal (3) or Fixed Defect (6). | Reversible defect (7) is highly associated with disease; Normal is linked to no disease. | High predictive power; requires categorical encoding. |



**Important Notes**
* **Strong Predictors:** BlockedVessels, ThalResult, ChestPain, ExAng.
* **Moderate Predictors:** STSlope, RestECG.
* **Weak Predictors:** Sex, FastingBS (clinically relevant but less statistically decisive).
* **Encoding Strategy:** One-Hot Encoding for nominal features; Ordinal Encoding for BlockedVessels.

## **3. Bivariate Analysis:**


### **3.1. Correlation Analysis**

In [ ]:
numeric_df = df_clean[['Age', 'Sex', 'ChestPain', 'RestBP', 'Cholesterol', 'FastingBS',
                      'RestECG', 'MaxHR', 'ExAng', 'STDepression', 'STSlope', 'BlockedVessels', 'ThalResult', 'Target']].copy()
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='RdBu_r', vmin=-1, vmax=1)

plt.show()

**Insight:**

| Interaction | Observation | Clinical Insight | Modeling Note |
| :--- | :--- | :--- | :--- |
| **STDepression vs. Target** | Patients with heart disease show significantly higher ST depression. | ST depression during exercise is a primary clinical indicator of myocardial ischemia. | Strong positive correlation; one of the most powerful predictors. |
| **MaxHR vs. Target** | Healthy patients achieve higher max heart rates; diseased patients peak much lower. | Low MaxHR is a red flag for poor cardiac response during physical stress. | Strong negative correlation; treat as a key physiological signal. |
| **Age & RestBP vs. Target** | Both show an upward trend with disease, but groups overlap significantly. | These are general risk factors rather than definitive standalone predictors. | Useful in combination with stronger features, but weak on their own. |
| **Blocked Vessels (ca) vs. Target** | Patients with 1–3 vessels blocked are almost always in the diseased group. | One of the most powerful categorical predictors of disease presence. | Should be prioritized during feature selection. |
| **Thalassemia (thal) vs. Target** | Reversible Defect = Disease; Normal flow = No Disease. | Clear separation between healthy and diseased groups based on stress test flow. | High predictive power; requires categorical encoding. |
| **Chest Pain (cp) vs. Target** | Asymptomatic patients are more likely to have disease than those with Typical Angina. | Highlights the clinical danger of "silent" heart issues. | Important categorical predictor for classification accuracy. |
| **ExAng vs. Target** | Most patients experiencing exercise-induced angina have heart disease. | Strong clinical marker for identifying symptomatic heart disease. | Binary feature; highly predictive of the positive class. |
| **MaxHR + STDepression** | Patients with high STDepression and low MaxHR cluster in the diseased group. | This interaction provides clearer separation than either feature alone. | Combining these features (interaction terms) can boost model accuracy. |



**Summary of Bivariate Insights**
* **Top Predictors:** STDepression, MaxHR, BlockedVessels, ThalResult, ChestPain, ExAng.
* **Risk Factors:** Age, RestBP, Cholesterol (weaker alone, stronger in combination).
* **Key Signal:** **Low MaxHR + High STDepression** = extremely strong indicator of disease.v

### **3.2. Numerical vs. Target**

In [ ]:
# Focused numerical features
numerical_features_to_compare = ['Cholesterol', 'MaxHR', 'STDepression']
n_rows = len(numerical_features_to_compare)
n_cols = 2  # One for KDE, one for Boxplot

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows), constrained_layout=True)

for i, col in enumerate(numerical_features_to_compare):
    # KDE Plot
    kde_plot = sns.kdeplot(
        data=df_plot, x=col, hue='Target_Label', fill=True,
        palette={'No Disease': 'skyblue', 'Disease': 'salmon'}, ax=axes[i, 0]
    )
    axes[i, 0].set_title(f'{col} Distribution by Heart Disease Status (KDE)', fontsize=14)
    axes[i, 0].set_xlabel(col, fontsize=12)
    axes[i, 0].set_ylabel('Density', fontsize=12)
    if kde_plot.get_legend() is not None:
        kde_plot.get_legend().set_title('Heart Disease Status')
    axes[i, 0].grid(axis='y', linestyle='--')

    # Boxplot
    sns.boxplot(
        data=df_plot, x='Target_Label', y=col,
        palette={'No Disease': 'skyblue', 'Disease': 'salmon'}, ax=axes[i, 1]
    )
    axes[i, 1].set_title(f'{col} Levels by Heart Disease Status (Boxplot)', fontsize=14)
    axes[i, 1].set_xlabel('Heart Disease Status', fontsize=12)
    axes[i, 1].set_ylabel(col, fontsize=12)
    axes[i, 1].grid(axis='y', linestyle='--')

fig.suptitle('Key Numerical Feature Distributions vs. Heart Disease Status', fontsize=18, y=1.02)
plt.show()



**Insight:**


| Feature | Observation | Clinical Insight | Modeling Note |
| :--- | :--- | :--- | :--- |
| **STDepression** | Diseased patients show significantly higher ST depression values. | Primary clinical marker of myocardial ischemia (oxygen struggle during exercise). | Strong positive correlation; outliers are clinically meaningful and must be retained. |
| **MaxHR** | Healthy patients achieve higher max heart rates; diseased patients peak lower. | Low MaxHR is a red flag for poor cardiac response and reduced exercise tolerance. | Strong negative correlation; one of the most powerful predictors in the dataset. |
| **Age** | Diseased patients tend to be older, though groups overlap significantly. | Age is a major risk factor but not a definitive standalone predictor. | Most effective when used in combination with stronger signals like STDepression. |
| **RestBP** | Higher values are more common in diseased patients, but overlap is high. | Hypertension increases risk but is not a standalone diagnostic marker. | Moderate predictor; scaling is recommended to align it with other features. |
| **Cholesterol** | Diseased patients often have higher levels, but many healthy individuals do too. | A "noisy" predictor; high values don't always guarantee disease presence. | Requires scaling/transformation; best used as part of a multi-feature risk profile. |


**Important Notes**
* **Strong Predictors:** **STDepression** (Positive correlation) and **MaxHR** (Negative correlation).
* **Moderate Predictors:** **Age, RestBP, and Cholesterol** (provide clinical context but are less decisive alone).
* **Key Takeaway:** The combination of **Low MaxHR + High STDepression** constitutes the strongest numerical signal for heart disease.

### **3.3. Categorical vs. Target**

In [ ]:
categorical_cols = ['Sex', 'ChestPain', 'FastingBS', 'RestECG', 'ExAng', 'STSlope', 'ThalResult', 'BlockedVessels']

# Create subplot grid: 4 rows × 2 columns
fig = make_subplots(rows=4, cols=2, subplot_titles=[f"{col} vs. Heart Disease" for col in categorical_cols])

# Loop through each categorical column
for i, col in enumerate(categorical_cols):
    row = (i // 2) + 1
    col_pos = (i % 2) + 1

    counts = df_plot.groupby([col, 'Target_Label']).size().reset_index(name='Count')

    fig.add_trace(
        go.Bar(
            x=counts[col].astype(str),
            y=counts['Count'],
            marker=dict(color=counts['Target_Label'].map({'No Disease':'#636EFA','Disease':'#EF553B'})),
            text=counts['Count'],
            textposition='auto',
            name=col
        ),
        row=row,
        col=col_pos
    )

# Update layout
fig.update_layout(
    height=1200,
    width=1000,
    title_text="Categorical Features vs. Heart Disease Status",
    title_font_size=22,
    showlegend=False,
    template='plotly_white'
)

fig.show()



**Insight:**


| Feature | Observation | Clinical Insight | Modeling Note |
| :--- | :--- | :--- | :--- |
| **Sex** | Males dominate the dataset (~68%); disease is more prevalent in males here. | Gender is a factor, but not decisive on its own in this specific dataset. | Weak predictor; potential for bias if the model overfits to male patterns. |
| **ChestPain (cp)** | Asymptomatic (Type 4) patients are more likely to have disease than those with Typical Angina. | "Silent" heart issues are dangerous—patients lacking obvious symptoms often have disease. | Strong predictor; requires One-Hot Encoding. |
| **FastingBS (fbs)** | The majority of patients (both healthy and diseased) have normal levels (≤120 mg/dl). | Linked to diabetes risk but is not a strong standalone predictor for heart disease. | Weak predictor; most useful when combined with cholesterol and blood pressure data. |
| **RestECG** | Most are Normal (0), but ST-T wave abnormalities or hypertrophy correlate with disease. | Abnormal resting ECG signals are clinically relevant markers for underlying issues. | Moderate predictor; requires categorical encoding. |
| **ExAng** | A clear split exists—most patients with exercise-induced angina are in the diseased group. | Angina during exercise is a significant clinical red flag. | Binary feature; highly predictive of disease presence. |
| **STSlope (slope)** | Most have a Flat slope (2); flat or downsloping segments are strongly associated with disease. | ST slope is a key physiological indicator of ischemia. | Important categorical predictor; requires encoding. |
| **BlockedVessels (ca)**| Patients with 1–3 blocked vessels are almost always diseased; 0 vessels is mostly healthy. | One of the strongest predictors; structural vessel damage is highly correlated with disease. | Ordinal feature; can be treated as numeric or categorical. |
| **ThalResult (thal)** | Reversible Defect (7) signals disease; Normal (3) signals no disease. | Blood flow tests provide a clear separation between healthy and diseased groups. | High predictive power; categorical encoding is mandatory. |


**Important Notes**
* **Strong Predictors:** BlockedVessels, ThalResult, ChestPain, and ExAng.
* **Moderate Predictors:** STSlope and RestECG.
* **Weak Predictors:** Sex and FastingBS (clinically relevant context, but less statistically decisive).
* **Encoding Strategy:**
    * **One-Hot:** ChestPain, ThalResult, RestECG, STSlope.
    * **Ordinal:** BlockedVessels.
    * **Binary:** Sex, ExAng.

### **3.4. Multivariate Analysis**

In [ ]:


# Example 1: Age vs MaxHR with regression line
plt.figure(figsize=(8,6))
sns.scatterplot(x='Age', y='MaxHR', hue='Target_Label', data=df_plot, palette='viridis', alpha=0.7)
sns.regplot(x='Age', y='MaxHR', data=df_plot, scatter=False, color='red', line_kws={'label':'Linear Trend'})
plt.title('Age vs Max Heart Rate (with Regression Line)', fontsize=14)
plt.xlabel('Age (Years)')
plt.ylabel('Maximum Heart Rate Achieved')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


# Example 3: MaxHR vs STDepression with regression line
plt.figure(figsize=(8,6))
sns.scatterplot(x='STDepression', y='MaxHR', hue='Target_Label', data=df_plot, palette='magma', alpha=0.7)
sns.regplot(x='STDepression', y='MaxHR', data=df_plot, scatter=False, color='blue', line_kws={'label':'Linear Trend'})
plt.title('Max Heart Rate vs ST Depression (with Regression Line)', fontsize=14)
plt.xlabel('ST Depression (mm)')
plt.ylabel('Maximum Heart Rate Achieved')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


**Insight:**


| Interaction | Observation | Clinical Insight | Modeling Note |
| :--- | :--- | :--- | :--- |
| **Age vs. MaxHR** | Younger patients achieve higher MaxHR; diseased patients cluster at lower MaxHR regardless of age. | Age naturally limits MaxHR, but a rate that is too low for a patient's age strongly signals disease. | Interaction is useful; older patients with low MaxHR represent the highest risk group. |
| **Cholesterol vs. RestBP** | Diseased patients tend toward higher levels in both, though significant overlap exists with healthy patients. | Both are cumulative risk factors; combined elevation increases the statistical likelihood of disease. | Noisy as standalone features; together they provide a more robust predictive signal. |
| **MaxHR vs. STDepression** | High STDepression combined with low MaxHR clusters almost exclusively in the diseased group. | This is the most decisive physiological signal, offering the clearest separation in the dataset. | Creating an interaction term (MaxHR × STDepression) can significantly boost model accuracy. |


**Important Notes**
* **Strongest Interaction:** **Low MaxHR + High STDepression** = almost guaranteed disease presence.
* **Supporting Interactions:**
    * **Age + MaxHR:** Highlights high-risk older patients with poor cardiac response.
    * **Cholesterol + RestBP:** Combined elevation increases risk profile.
* **Modeling Strategy:**
    * Use **interaction features** (especially for MaxHR and STDepression).
    * **Scale features** (Age, Cholesterol, RestBP, MaxHR, STDepression) before training.
    * **Retain outliers**, as they represent clinically meaningful high-risk signals rather than data errors.

---

### **Final Conclusion for Modeling**

- **Key predictors:** BlockedVessels (`ca`), MaxHR (`thalach`), STDepression (`oldpeak`), ThalResult (`thal`), ChestPain (`cp`).  
- **Target:** Convert multi-class `num` into binary (0 = no disease, >0 = disease).  
- **Prep needs:** Impute missing (`ca`, `thal`), scale continuous (`thalach`, `oldpeak`), encode categorical (`cp`, `thal`).  
- **Modeling:** Begin with interpretable baselines (logistic regression, decision tree), then move to ensembles (RF, GBM).  
- **Evaluation:** Use stratified CV, AUC, F1, precision/recall; apply cost-sensitive metrics from “Costs” folder.  
- **Next steps:** Finalize preprocessing pipeline, run baseline + ensemble experiments, report feature importance and clinical interpretability.  

**Conclusion:** With proper preprocessing and cost-aware evaluation, these features enable robust, clinically meaningful heart disease prediction.

# **Data Cleaning**

#### **Imputation Decision: Mode Selected**

In [ ]:
cols_to_impute = ['BlockedVessels', 'ThalResult']
df_copy = df.copy()
for col in cols_to_impute:
    df_copy[col] = df_copy[col].fillna(df[col].mode()[0])

### **Why Mode Imputation?**

1. Preserves data type integrity  
   - `BlockedVessels`: integers (0–3)  
   - `ThalResult`: categories (3, 6, 7)  
   - Mean → invalid decimals (e.g., 1.67 vessels)

2. Clinical validity  
   - Medical values must remain realistic  
   - Mode keeps imputed values within valid ranges

3. Distribution preservation  
   - Maintains original shape  
   - Minimal impact on stats

4. Comparison  
   - Backfill/Forward Fill → arbitrary  
   - Mean → invalid values  
   - Mode → correct & clinically sound

**Result:** Use **Mode Imputation** for `BlockedVessels` and `ThalResult`.


# **4. Feature Selection**

Now that we've completed our EDA and data cleaning, we need to identify which features are most predictive of heart disease.

**Feature Selection Methods:**
1. **Correlation Analysis** - Identify features with strong linear relationships to the target
2. **Mutual Information** - Capture non-linear relationships
3. **Chi-Square Test** - Statistical significance for categorical features


In [ ]:
# Create a copy for feature selection
df_fs = df_copy.copy()

# Create binary target (0 = No Disease, 1 = Disease)
df_fs['Target_Binary'] = (df_fs['Target'] > 0).astype(int)

# Separate features and target
X = df_fs.drop(['Target', 'Target_Binary'], axis=1)
y = df_fs['Target_Binary']

print(f"Dataset shape: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"\nTarget distribution:")
print(y.value_counts())

## **4.1. Correlation-Based Feature Selection**

We'll examine the correlation between each feature and the target variable to identify strong linear relationships.

In [ ]:
# Calculate correlation with target
correlations = df_fs.drop('Target_Binary', axis=1).corrwith(df_fs['Target_Binary']).abs()
correlations = correlations.sort_values(ascending=False)

# Visualize correlations
plt.figure(figsize=(10, 8))
correlations.plot(kind='barh', color='steelblue')
plt.title('Absolute Correlation with Heart Disease (Target)', fontsize=14, fontweight='bold')
plt.xlabel('Absolute Correlation Coefficient', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.axvline(x=0.3, color='red', linestyle='--', label='Threshold (0.3)')
plt.legend()
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(correlations):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nCorrelation Rankings:")
print(correlations)

# Select features with correlation > 0.3
correlation_features = correlations[correlations > 0.3].index.tolist()
if 'Target' in correlation_features:
    correlation_features.remove('Target')
print(f"\nFeatures selected by correlation (> 0.3): {correlation_features}")

## **4.2. Mutual Information**

Mutual Information captures both linear and non-linear relationships between features and the target.


In [ ]:
# Calculate mutual information scores
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_scores = pd.Series(mi_scores, index=X.columns).sort_values(ascending=False)

# Visualize mutual information
plt.figure(figsize=(10, 8))
mi_scores.plot(kind='barh', color='coral')
plt.title('Mutual Information Scores', fontsize=14, fontweight='bold')
plt.xlabel('Mutual Information Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(mi_scores):
    plt.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nMutual Information Rankings:")
print(mi_scores)

# Select top 10 features by MI
mi_features = mi_scores.nlargest(10).index.tolist()
print(f"\nTop 10 features by Mutual Information: {mi_features}")

## **4.3. Chi-Square Test**

Chi-Square test measures the dependence between categorical features and the target variable.


In [ ]:
# For Chi-Square, we need non-negative values
# Scale features to be non-negative
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Calculate Chi-Square scores
chi2_scores, p_values = chi2(X_scaled, y)
chi2_results = pd.DataFrame({
    'Feature': X.columns,
    'Chi2_Score': chi2_scores,
    'P_Value': p_values
}).sort_values('Chi2_Score', ascending=False)

# Visualize Chi-Square scores
plt.figure(figsize=(10, 8))
plt.barh(chi2_results['Feature'], chi2_results['Chi2_Score'], color='mediumseagreen')
plt.title('Chi-Square Test Scores', fontsize=14, fontweight='bold')
plt.xlabel('Chi-Square Score', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(chi2_results.iterrows()):
    plt.text(row['Chi2_Score'] + 1, i, f"{row['Chi2_Score']:.1f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\nChi-Square Test Results:")
print(chi2_results)

# Select features with p-value < 0.05
chi2_features = chi2_results[chi2_results['P_Value'] < 0.05]['Feature'].tolist()
print(f"\nStatistically significant features (p < 0.05): {chi2_features}")


### **4.4. Combined Feature Selection Results**
Let's combine all methods to identify the most consistently important features.

In [ ]:
# Create a voting system - count how many methods selected each feature
all_features = list(X.columns)
feature_votes = {feature: 0 for feature in all_features}

# Count votes from each method
for feature in correlation_features:
    if feature in feature_votes:
        feature_votes[feature] += 1

for feature in mi_features:
    if feature in feature_votes:
        feature_votes[feature] += 1

for feature in chi2_features:
    if feature in feature_votes:
        feature_votes[feature] += 1

# Create summary DataFrame
feature_summary = pd.DataFrame({
    'Feature': list(feature_votes.keys()),
    'Votes': list(feature_votes.values()),
    'Correlation': [feature in correlation_features for feature in feature_votes.keys()],
    'Mutual_Info': [feature in mi_features for feature in feature_votes.keys()],
    'Chi2': [feature in chi2_features for feature in feature_votes.keys()],
}).sort_values('Votes', ascending=False)

print("="*80)
print("FEATURE SELECTION SUMMARY")
print("="*80)
print(feature_summary)

# Visualize voting results
plt.figure(figsize=(12, 8))
colors = plt.cm.RdYlGn(feature_summary['Votes'] / 3)  # Color by number of votes
plt.barh(feature_summary['Feature'], feature_summary['Votes'], color=colors)
plt.title('Feature Selection Voting Results\n(Number of Methods Selecting Each Feature)',
          fontsize=14, fontweight='bold')
plt.xlabel('Number of Methods (out of 3)', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='Threshold (3/5 methods)')
plt.legend()
plt.grid(axis='x', alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(feature_summary.iterrows()):
    plt.text(row['Votes'] + 0.1, i, f"{int(row['Votes'])}/3",
             va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

### **4.5. Final Feature Selection**
Based on the voting results, we'll select features that were chosen by at least 3 out of 3 methods.

In [ ]:
# Select features with at least 3 votes
VOTE_THRESHOLD = 3
selected_features = feature_summary[feature_summary['Votes'] >= VOTE_THRESHOLD]['Feature'].tolist()

print("="*80)
print("FINAL SELECTED FEATURES")
print("="*80)
print(f"\nThreshold: {VOTE_THRESHOLD}/3 methods")
print(f"Number of features selected: {len(selected_features)}")
print(f"\nSelected features:")
for i, feature in enumerate(selected_features, 1):
    votes = feature_summary[feature_summary['Feature'] == feature]['Votes'].values[0]
    print(f"  {i}. {feature} ({votes}/3 votes)")

# Create final feature set
X_selected = X[selected_features]
print(f"\nFinal feature matrix shape: {X_selected.shape}")

# Save selected features for modeling
selected_features_dict = {
    'features': selected_features,
    'vote_threshold': VOTE_THRESHOLD,
    'total_methods': 3
}

print("\n" + "="*80)
print("Feature selection complete!")
print("="*80)

## **5. Data Preprocessing**

In [ ]:
# Create a new copy for preprocessing
df_prep = df_copy.copy()

# Remove irrelevant features
features_to_remove = ['Age','Sex','RestBP','Cholesterol','FastingBS','RestECG','STSlope']
df_prep.drop(columns = features_to_remove, inplace=True)
df_prep.info()

**One-hot Encoding**

In [ ]:
#One-hot Encoding

# Mapping values for "thal" and "cp" columns
thal_mapping = {3: 'normal', 6: 'fixed defect', 7: 'reversible defect'}
cp_mapping = {1: 'typical angina', 2: 'atypical angina', 3: 'non-anginal pain', 4: 'asymptomatic'}

# Applying mapping to "thal" and "cp" columns
df_prep['ThalResult'] = df_prep['ThalResult'].map(thal_mapping)
df_prep['ChestPain'] = df['ChestPain'].map(cp_mapping)

# Perform one-hot encoding
df_prep = pd.get_dummies(df_prep, columns=['ThalResult', 'ExAng', 'ChestPain', 'BlockedVessels'])

df_prep.info()
df_prep.head()

**Normalization**

In [ ]:
# Normalise quantitative variables

# Extracting continuous coloumns
continuous_columns = ['MaxHR','STDepression']

# MinMax Normalisation
scaler = MinMaxScaler()
df_prep[continuous_columns] = scaler.fit_transform(df[continuous_columns])

df_prep.head()
df_prep.info()

**ColumnTransformer**

In [ ]:
# Define the feature types based on X_selected and Final Conclusion for Modeling
# X_selected contains: ['ChestPain', 'STDepression', 'STSlope', 'BlockedVessels', 'ExAng', 'ThalResult']
numeric_features = ['STDepression']
binary_features = ['ExAng'] # Already 0/1, so passthrough
ordinal_features = ['BlockedVessels'] # Ordinal values 0, 1, 2, 3
nominal_features = ['ChestPain', 'ThalResult', 'STSlope'] # These will be one-hot encoded

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("bin", "passthrough", binary_features),
        ("ord", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), ordinal_features),
        ("cat", OneHotEncoder(handle_unknown='ignore'), nominal_features)
    ],
    remainder='passthrough' # Keep any other columns not explicitly transformed
)

**Train / Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# **Modeling**

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(
        eval_metric="logloss",
        use_label_encoder=False,
        random_state=42
    ),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

**Evaluation**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_classification(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_proba)
    }


**Train the models**

In [ ]:
results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results[name] = evaluate_classification(y_test, y_pred, y_proba)


**Pipeline**

In [ ]:
Pipeline([
    ("preprocessor", preprocessor),
    ("model",  model)
])

**Best Model**

In [ ]:
best_model_name = max(results, key=lambda x: results[x]["Recall"])
print(f"Best Model: {best_model_name}")

**Model Comparison**

In [ ]:
from sklearn.metrics import confusion_matrix, RocCurveDisplay

pipes = {}
for name, model in models.items():
    pipe = Pipeline(
        [
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    try:
        y_proba = pipe.predict_proba(X_test)[:, 1]
    except Exception:
        try:
            y_score = pipe.decision_function(X_test)

            y_proba = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-9)
        except Exception:
            y_proba = np.zeros_like(y_pred, dtype=float)

    results[name] = evaluate_classification(y_test, y_pred, y_proba)
    pipes[name] = pipe


results_df = pd.DataFrame(results).T.sort_values(by="Recall", ascending=False)
print("\n===COMPARING MODELS ON TEST SET===")
print(results_df.round(4))

# ========== 8) Confusion Matrices ==========
def plot_confusion_matrices(models_dict, pipes_dict, X_test, y_test, normalized=False):
    n_models = len(models_dict)
    cols = 2
    rows = int(np.ceil(n_models / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, 4*rows))
    axes = axes.flatten()

    for i, (name, _) in enumerate(models_dict.items()):
        ax = axes[i]
        pipe = pipes_dict[name]
        y_pred = pipe.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        if normalized:
            cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
            fmt = ".2f"
            title = f"{name} - Confusion Matrix (Normalized)"
        else:
            fmt = "d"
            title = f"{name} - Confusion Matrix"

        sns.heatmap(cm, annot=True, fmt=fmt, cmap="Blues", cbar=False,
                    xticklabels=["No Disease (0)", "Disease (1)"],
                    yticklabels=["No Disease (0)", "Disease (1)"],
                    ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title(title)


    for j in range(i+1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

print("\n[Confusion Matrices] Raw Values:")
plot_confusion_matrices(models, pipes, X_test, y_test, normalized=False)

print("\n[Confusion Matrices](Normalized):")
plot_confusion_matrices(models, pipes, X_test, y_test, normalized=True)


In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1"]


ordered = results_df.sort_values(by="Recall", ascending=False).copy()


plot_df = ordered[metrics_to_plot].copy()
plot_df["Model"] = ordered.index
plot_df = plot_df.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(12, 6))
sns.barplot(
    data=plot_df, x="Model", y="Score", hue="Metric",
    palette="Set2"
)
plt.title("model Comparison", fontsize=14)
plt.xlabel("model", fontsize=12)
plt.ylabel("value", fontsize=12)
plt.ylim(0, 1)
plt.legend(title="standrad", loc="upper right")
plt.grid(axis="y", ls="--", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
auc_results = {}
pipes = {}

plt.figure(figsize=(8, 6))
for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    pipes[name] = pipe


    try:
        y_proba = pipe.predict_proba(X_test)[:, 1]
    except Exception:
        try:
            y_score = pipe.decision_function(X_test)
            y_proba = (y_score - y_score.min()) / (y_score.max() - y_score.min() + 1e-9)
        except Exception:
            print(f"[Warning] {name} does not support predict_proba/decision_function, it will be skipped in ROC.")
            continue

    auc = roc_auc_score(y_test, y_proba)
    auc_results[name] = auc

    RocCurveDisplay.from_predictions(y_test, y_proba, name=f"{name} (AUC={auc:.3f})")

plt.title("ROC Curves for Models")
plt.grid(True, ls="--", alpha=0.5)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

auc_df = pd.Series(auc_results, name="ROC_AUC").sort_values(ascending=False)
print("\n=== ROC-AUC Comparison (Test Set) ===")
print(auc_df.round(4))

plt.figure(figsize=(8, 5))
sns.barplot(x=auc_df.index, y=auc_df.values, palette="viridis")
plt.title("Comparison of Models by ROC-AUC", fontsize=14)
plt.xlabel("Model", fontsize=12)
plt.ylabel("ROC-AUC", fontsize=12)
plt.ylim(0.5, 1.0)

for i, v in enumerate(auc_df.values):
    plt.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_auc_table = {}
for name, pipe in pipes.items():
    base_model = models[name]
    cv_pipe = Pipeline([("preprocessor", preprocessor), ("model", base_model)])
    cv_scores = cross_val_score(cv_pipe, X, y, cv=cv, scoring="roc_auc")
    cv_auc_table[name] = {
        "AUC_mean": np.mean(cv_scores),
        "AUC_std": np.std(cv_scores),
        "Scores": cv_scores
    }

cv_auc_df = pd.DataFrame(cv_auc_table).T.sort_values(by="AUC_mean", ascending=False)
print("\n=== Cross-Validated ROC-AUC (5-Fold) ===")
print(cv_auc_df[["AUC_mean", "AUC_std"]].round(4))


## **Conclusion**

The results demonstrated that **Logistic Regression** was the most suitable model, achieving the highest **Recall**, which is critical in the medical domain to minimize false negatives (undiagnosed patients). Key predictive features identified include ST depression during exercise, number of blocked vessels, chest pain type, and thallium test results.


## **Recommendations**

1.  **Adopt Logistic Regression as the primary model** for clinical decision support, prioritizing high sensitivity (Recall) even at the expense of slight accuracy reduction.
2.  **Expand the dataset** with more recent and diverse patient records to improve model generalization.
3.  **Integrate the model into healthcare decision-support systems** to alert physicians when a patient shows high risk.
4.  **Include additional features** such as lifestyle factors and family history for better predictive performance.
5.  **Use the model as an assistive tool, not a replacement for clinical diagnosis**, ensuring it complements traditional medical assessments.

---

In [ ]:
for name, model_pipeline in pipes.items():
    print(f"\n--- Testing pipeline for model: {name} ---")

    # Test pipeline integrity
    test_pipeline_used(model_pipeline)
    test_column_transformer_used(model_pipeline)
    test_categorical_encoding(model_pipeline)
    test_feature_scaling(model_pipeline)
    test_no_data_leakage(model_pipeline)

    # Get predictions for classification metrics test
    y_pred_model = model_pipeline.predict(X_test)

    # Handle models that might not have predict_proba (e.g., if a custom model without it was added)
    try:
        y_proba_model = model_pipeline.predict_proba(X_test)[:, 1]
    except AttributeError:
        # Fallback for models without predict_proba if needed, though for standard classifiers, it's usually present
        y_proba_model = np.zeros_like(y_pred_model, dtype=float) # Placeholder, might not pass ROC_AUC test if used
        print(f"Warning: Model {name} does not have predict_proba. ROC_AUC might be affected.")

    # Evaluate metrics for the current model
    metrics_model = evaluate_classification(y_test, y_pred_model, y_proba_model)

    # Test classification metrics
    test_classification_metrics(metrics_model)

    print(f"All pipeline and model integrity tests passed for {name}!")

print("\n--- All model pipelines have been tested. ---")